In [1]:
!pip install sacremoses kaggle


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.5/897.5 kB 13.3 MB/s eta 0:00:00 0:00:01


In [2]:
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import ViTImageProcessor, VisionEncoderDecoderModel
from PIL import Image
import os
from glob import glob
from tqdm import tqdm
import warnings
import gc
import random

warnings.filterwarnings("ignore")

# OPTIMIZATION: Deterministic runs & fast convolutions
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)
torch.backends.cudnn.benchmark = True 

print("==================================================")
print("🚀 PHASE 1: HYBRID FUSION EXTRACTOR (PRODUCTION)")
print("==================================================")

# 1. PATH CONFIGURATION
BASE_IMAGE_DIR = '/kaggle/input/datasets/organizations/nih-chest-xrays/data'
DATA_CSV_PATH = '/kaggle/input/datasets/organizations/nih-chest-xrays/data/Data_Entry_2017.csv'
WEIGHTS_PATH = '/kaggle/input/datasets/tarak577/reflective-cxr-lora-weights/fused_model_epoch_10'

# 2. IMAGE INDEXING
print("Indexing dataset file paths...")
image_map = {os.path.basename(x): x for x in glob(os.path.join(BASE_IMAGE_DIR, "images_*", "images", "*.png"))}

df = pd.read_csv(DATA_CSV_PATH)
df['Is_Real_Abnormal'] = (df['Finding Labels'] != "No Finding").astype(int)
df['img_path'] = df['Image Index'].map(image_map)
df = df.dropna(subset=['img_path']).reset_index(drop=True)
TOTAL_IMAGES = len(df)
print(f"Total valid images mapped: {TOTAL_IMAGES}")

# 3. DATASET & SAFE COLLATOR
class CXRFeatureDataset(Dataset):
    def __init__(self, df, processor):
        self.df = df
        self.processor = processor

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        try:
            # OPTIMIZATION: convert("RGB") inherently loads the image buffer safely
            image = Image.open(row['img_path']).convert("RGB")
            pv = self.processor(images=image, return_tensors="pt").pixel_values.squeeze(0)
            return pv, str(row['Image Index']), int(row['Is_Real_Abnormal'])
        except Exception:
            return None

def safe_collate(batch):
    # OPTIMIZATION: Clean Pythonic list comprehension
    batch = [b for b in batch if b is not None]
    if not batch: return None, None, None
    
    pvs = torch.stack([b[0] for b in batch])
    img_ids = [b[1] for b in batch]
    labels = [b[2] for b in batch]
    return pvs, img_ids, labels

# 4. INITIALIZATION
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
processor = ViTImageProcessor.from_pretrained("google/vit-base-patch16-224")

print("Loading Encoder & Flushing VRAM...")
vlm = VisionEncoderDecoderModel.from_pretrained(WEIGHTS_PATH)
vit_encoder = vlm.get_encoder().to(device)
vit_encoder.eval()

del vlm
gc.collect()
torch.cuda.empty_cache()

for p in vit_encoder.parameters(): p.requires_grad = False

# OPTIMIZATION: Aggressive batch size (32) for T4/P100 inference
loader = DataLoader(
    CXRFeatureDataset(df, processor), batch_size=32, shuffle=False, 
    num_workers=6, pin_memory=True, collate_fn=safe_collate, persistent_workers=True
)

# 5. PREALLOCATE MEMORY
print(f"Preallocating RAM for features: {TOTAL_IMAGES} x 2304...")
feature_cache = np.zeros((TOTAL_IMAGES, 2304), dtype=np.float32)
id_cache = np.empty(TOTAL_IMAGES, dtype=object)
label_cache = np.empty(TOTAL_IMAGES, dtype=np.int8)

# 6. EXTRACTION LOOP
current_idx = 0
print("Starting High-Speed Feature Extraction...")

with torch.inference_mode():
    for batch_data in tqdm(loader, desc="Extracting"):
        if batch_data[0] is None: 
            continue
        
        pvs, img_ids, labels = batch_data
        pvs = pvs.to(device, non_blocking=True)
        batch_size = pvs.size(0)
        
        with torch.amp.autocast('cuda'):
            outputs = vit_encoder(pixel_values=pvs)
            hidden_states = outputs.last_hidden_state
            
            cls_token = hidden_states[:, 0, :]
            patch_tokens = hidden_states[:, 1:, :]
            mean_pool = patch_tokens.mean(dim=1)
            max_pool = patch_tokens.max(dim=1).values
            
            fused = torch.cat([cls_token, mean_pool, max_pool], dim=1)
            
            del outputs, hidden_states, cls_token, patch_tokens, mean_pool, max_pool
        
        feature_cache[current_idx : current_idx + batch_size] = fused.cpu().numpy()
        id_cache[current_idx : current_idx + batch_size] = img_ids
        label_cache[current_idx : current_idx + batch_size] = labels
        
        del fused, pvs
        current_idx += batch_size
        
        # OPTIMIZATION: Deep cache flush and explicit progress logging
        if current_idx % 5000 == 0:
            print(f"[{current_idx}/{TOTAL_IMAGES}] VRAM check & GC sweep...")
            torch.cuda.empty_cache()
            gc.collect()

# 7. TRIM ARRAYS & VALIDATE
feature_cache = feature_cache[:current_idx]
id_cache = id_cache[:current_idx]
label_cache = label_cache[:current_idx]

assert feature_cache.shape[0] == len(id_cache) == len(label_cache), "Dimension mismatch detected!"

# 8. SAVE ARTIFACTS
print(f"\nExtraction complete! Saving {current_idx} valid feature vectors...")

# OPTIMIZATION: Direct save without redundant astype memory duplication
np.save("v12_fusion_features.npy", feature_cache)
np.save("v12_fusion_ids.npy", id_cache)
np.save("v12_fusion_labels.npy", label_cache)

print("✅ PHASE 1 COMPLETE. Arrays saved securely.")

🚀 PHASE 1: HYBRID FUSION EXTRACTOR (PRODUCTION)
Indexing dataset file paths...
Total valid images mapped: 112120


preprocessor_config.json:   0%|          | 0.00/160 [00:00<?, ?B/s]

Loading Encoder & Flushing VRAM...


Loading weights:   0%|          | 0/591 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie decoder.biogpt.embed_tokens.weight to decoder.output_projection.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Preallocating RAM for features: 112120 x 2304...
Starting High-Speed Feature Extraction...


... (8 outputs left out)

Extracting:  89%|████████▉ | 3124/3504 [16:50<02:31,  2.51it/s]

[100000/112120] VRAM check & GC sweep...


Extracting: 100%|██████████| 3504/3504 [18:50<00:00,  3.10it/s]



Extraction complete! Saving 112120 valid feature vectors...
✅ PHASE 1 COMPLETE. Arrays saved securely.


In [4]:
import shutil
import os
from IPython.display import FileLink

print("Compressing the 1.3GB features file...")
# Zip the file to reduce size and help the browser download manager
shutil.make_archive('v12_fusion_backup', 'zip', '/kaggle/working', 'v12_fusion_features.npy')

# Also zip the metadata so you have everything
shutil.make_archive('v12_metadata', 'zip', '/kaggle/working', 'v12_fusion_ids.npy')
shutil.make_archive('v12_labels', 'zip', '/kaggle/working', 'v12_fusion_labels.npy')

print("Generating direct download links. Click them below:")
display(FileLink('v12_fusion_backup.zip'))
display(FileLink('v12_metadata.zip'))
display(FileLink('v12_labels.zip'))

Compressing the 1.3GB features file...
Generating direct download links. Click them below:


/kaggle/working/v12_fusion_backup.zip

/kaggle/working/v12_metadata.zip

/kaggle/working/v12_labels.zip